In [9]:
import sys

assert sys.version_info >= (3,10)

In [10]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [11]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [12]:
from pathlib import Path

IMAGES_PATH =  Path() / "images" / "Physics_Informed_Digital_Twin_for_Hydraulic_Systems_Navier_Stokes_3D"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension='png', tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [13]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


In [14]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Defining physical parameters

In [15]:
rho = 1.0
mu = 0.01

Define a 3D cuboid Domain

In [16]:
xmin, xmax = [0.0, -0.5, -0.5], [2.0, 0.5, 0.5]
geom = dde.geometry.Cuboid(xmin=xmin, xmax=xmax)

Generating mock Sensor Data

In a real scenario, you can replace these with your actual physical experimental measurements.

In [17]:
num_sensors = 50

Randomly scatter 50 Sensors throughout the interior of the channel

In [18]:
np.random.seed(42)
sensor_x = np.random.uniform(0.1, 1.9, (num_sensors, 1))
sensor_y = np.random.uniform(-0.4, 0.4, (num_sensors, 1))
sensor_z = np.random.uniform(-0.4, 0.4, (num_sensors, 1))
sensor_coords  = np.hstack((sensor_x, sensor_y, sensor_z))


Let's assume a dummy decay function for demostration

Y_True Components: [ u, v, w, p]

In [19]:
mock_u = 1.0 * (1.0 -(sensor_y/0.5)**2) * (1.0 - (sensor_z/0.5)**2)
mock_v = np.zeros_like(sensor_x)
mock_w = np.zeros_like(sensor_x)
mock_p = 0.1 * (2.0 - sensor_x) # Linear pressure drop guess

sensor_values = np.hstack((mock_u, mock_v, mock_w, mock_p))

Preparing data for Deepxde PointSetBC points

components 0=u, 1=v, 2=w, 3=p

In [20]:
observe_u = dde.icbc.PointSetBC(sensor_coords, sensor_values[:, 0:1], component=0)
observe_v = dde.icbc.PointSetBC(sensor_coords, sensor_values[:, 1:2], component=1)
observe_w = dde.icbc.PointSetBC(sensor_coords, sensor_values[:, 2:3], component=2)
observe_p = dde.icbc.PointSetBC(sensor_coords, sensor_values[:, 3:4], component=3)

Setting up the Boundary Conditions

In [21]:
def is_inlet(x, on_boundary):
    return on_boundary and np.isclose(x[0], xmin[0])

def is_outlet(x, on_boundary):
    return on_boundary and np.isclose(x[0], xmax[0])

def is_wall(x, on_boundary):
    return on_boundary and not np.isclose(x[0], xmin[0]) and not np.isclose(x[0], xmax[0])

def inlet_velocity_profile(x):
    y_coord = x[:, 1:2]
    z_coord = x[:, 2:3]
    return 1.0 * (1.0 - (y_coord / 0.5)**2) * (1.0 - (z_coord / 0.5)**2)

bc_inlet_u = dde.icbc.DirichletBC(geom, inlet_velocity_profile, is_inlet, component=0)
bc_inlet_v = dde.icbc.DirichletBC(geom, lambda x: 0.0, is_inlet, component=1)
bc_inlet_w = dde.icbc.DirichletBC(geom, lambda x: 0.0, is_inlet, component=2)

bc_wall_u = dde.icbc.DirichletBC(geom, lambda x: 0, is_wall, component=0)
bc_wall_v = dde.icbc.DirichletBC(geom, lambda x: 0, is_wall, component=1)
bc_wall_w = dde.icbc.DirichletBC(geom, lambda x:0, is_wall, component=2)

bc_outlet_p = dde.icbc.DirichletBC(geom, lambda x: 0, is_outlet, component=3)


Consolidate All Conditions

Combine analytical boundaries and physical sensor constraints together

In [22]:
all_conditions = [
    bc_inlet_u, bc_inlet_v, bc_inlet_w,
    bc_wall_u, bc_wall_v, bc_wall_w,
    bc_outlet_p,
    observe_u, observe_v, observe_w, observe_p
]

Define a function for Navier-Stokes-3D equation

In [23]:
def navier_stokes_3d(x, y):
    u, v, w, p = y[:, 0:1], y[:, 1:2], y[:, 2:3], y[:, 3:4]

    du_dx = dde.grad.jacobian(y, x, i=0, j=0)
    du_dy = dde.grad.jacobian(y, x, i=0, j=1)
    du_dz = dde.grad.jacobian(y, x, i=0, j=2)
    dv_dx = dde.grad.jacobian(y, x, i=1, j=0)
    dv_dy = dde.grad.jacobian(y, x, i=1, j=1)
    dv_dz = dde.grad.jacobian(y, x, i=1, j=2)
    dw_dx = dde.grad.jacobian(y,x, i=2, j=0)
    dw_dy = dde.grad.jacobian(y, x, i=2, j=1)
    dw_dz = dde.grad.jacobian(y, x, i=2, j=2)

    dp_dx = dde.grad.jacobian(y, x, i=3, j=0)
    dp_dy = dde.grad.jacobian(y, x, i=3, j=1)
    dp_dz = dde.grad.jacobian(y, x, i=3, j=2)

    du_dxx = dde.grad.hessian(y, x, component=0, i=0, j=0)
    du_dyy = dde.grad.hessian(y, x, component=0, i=1, j=1)
    du_dzz = dde.grad.hessian(y, x, component=0, i=2, j=2)
    dv_dxx = dde.grad.hessian(y, x, component=1, i=0, j=0)
    dv_dyy = dde.grad.hessian(y, x, component=1, i=1, j=1)
    dv_dzz = dde.grad.hessian(y, x, component=1, i=2, j=2)
    dw_dxx = dde.grad.hessian(y, x, component=2, i=0, j=0)
    dw_dyy = dde.grad.hessian(y, x, component=2, i=1, j=1)
    dw_dzz = dde.grad.hessian(y, x, component=2, i=2, j=2)

    x_momentum = rho * (u * du_dx + v * du_dy + w * du_dz) + dp_dx - mu * (du_dxx + du_dyy + du_dzz)
    y_momentum = rho * (u * dv_dx + v * dv_dy + w * dv_dz) + dp_dy - mu * (dv_dxx + dv_dyy + dv_dzz)
    z_momentum = rho * (u * dw_dx + v * dw_dy + w * dw_dz) + dp_dz - mu * (dw_dxx + dw_dyy + dw_dzz)
    continuity = du_dx + dv_dy + dw_dz

    return [x_momentum, y_momentum, z_momentum, continuity]


Combine all the for compiling

In [24]:
data = dde.data.PDE(
    geom, navier_stokes_3d, all_conditions,
    num_domain=3000,
    num_boundary=600,
    anchors=sensor_coords,
    num_test=600
)

Build a Neural Network and Create a Model

In [29]:
net = dde.nn.FNN([3, 256, 128, 64, 4], "tanh", "Glorot uniform")
model = dde.Model(data, net)


In [30]:
class SoftmaxAdaptiveWeights(dde.callbacks.Callback):
    def __init__(self, model, every_n_epochs=100, alpha_0=1.0, gamma=0.999):
        super().__init__()
        self.model = model
        self.every_n_epochs = every_n_epochs
        self.alpha = alpha_0
        self.gamma = gamma
        self.epoch = 0

    def on_epoch_end(self):
        self.epoch += 1
        if self.epoch % self.every_n_epochs != 0:
            return
        if len(self.model.losshistory.loss_train) == 0:
            return
        current_losses = self.model.losshistory.loss_train[-1]
        exp_terms = np.exp(self.alpha * np.array(current_losses))
        new_weights = exp_terms / (np.sum(exp_terms)+1e-8)
        self.alpha *= self.gamma
        self.model.loss_weights = new_weights.tolist()
        # print(f"Epoch {self.epoch}: Adaptive weights updated to {new_weights.tolist()}, alpha decayed to {self.alpha:.4f}")

adaptive_callback = SoftmaxAdaptiveWeights(model, every_n_epochs=20)

In [31]:
loss_weights = [1, 1, 1, 2] + [1]*7 + [5, 5, 5, 5]

Train the Model

In [ ]:
print("\nStage 1: Adam Optimizer")
model.compile(
    "adam", lr=0.001,
    loss_weights=loss_weights,
    decay=("inverse time", 1000, 0.1)
)
losshistory, train_state = model.train(iterations=1000, display_every=200, callbacks=[adaptive_callback])
dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights=model.loss_weights)
losshistory, train_state = model.train(display_every=50, callbacks=[adaptive_callback])


Stage 1: Adam Optimizer
Compiling model...
'compile' took 0.000464 s

Training model...

Step      Train loss                                                                                                                                                Test loss                                                                                                                                                 Test metric
0         [5.97e-02, 3.10e-03, 2.35e-03, 8.64e-03, 2.92e-01, 7.02e-04, 2.71e-03, 7.84e-02, 6.31e-02, 5.28e-03, 1.85e-01, 9.82e-01, 2.71e-01, 8.81e-03, 1.90e-01]    [6.05e-02, 2.82e-03, 2.37e-03, 8.61e-03, 2.92e-01, 7.02e-04, 2.71e-03, 7.84e-02, 6.31e-02, 5.28e-03, 1.85e-01, 9.82e-01, 2.71e-01, 8.81e-03, 1.90e-01]    []  
200       [2.65e-04, 1.08e-06, 1.27e-06, 9.58e-06, 6.09e-03, 3.02e-06, 1.70e-06, 1.06e-02, 3.57e-06, 3.34e-06, 5.22e-05, 9.53e-03, 2.93e-06, 1.49e-06, 5.77e-05]    [2.64e-04, 7.01e-07, 1.03e-06, 7.51e-06, 6.09e-03, 3.02e-06, 1.70e-06, 1.06e-02, 3.57e-06, 3.

In [ ]:
np.random.seed(45)
sensor_x = np.random.uniform(0.1, 1.9, (num_sensors, 1))
sensor_y = np.random.uniform(-0.4, 0.4, (num_sensors, 1))
sensor_z = np.random.uniform(-0.4, 0.4, (num_sensors, 1))
eval_coords  = np.hstack((sensor_x, sensor_y, sensor_z))

In [ ]:
predictions = model.predict(eval_coords)

In [ ]:
mock_u = 1.0 * (1.0 -(sensor_y/0.5)**2) * (1.0 - (sensor_z/0.5)**2)
mock_v = np.zeros_like(sensor_x)
mock_w = np.zeros_like(sensor_x)
mock_p = 0.1 * (2.0 - sensor_x) # Linear pressure drop guess

eval_values = np.hstack((mock_u, mock_v, mock_w, mock_p))

In [ ]:
u_pred, v_pred, w_pred, p_pred = predictions[:, 0], predictions[:, 1], predictions[:, 2], predictions[:, 3]
u_true, v_true, w_true, p_true = eval_values[:, 0], eval_values[:, 1], eval_values[:, 2], eval_values[:, 3]


Compute Relative L2 error

In [ ]:
def relative_l2_error(y_true, y_pred):
    return np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)

# Compute L2 errors (safely excluding cross-flow velocities if they are perfectly 0)
u_l2 = relative_l2_error(u_true, u_pred)
p_l2 = relative_l2_error(p_true, p_pred)
u_mae = np.mean(np.abs(u_true - u_pred))

print(f"Evaluation Results across {len(eval_coords)} Test Sensors:")
print(f" -> Velocity u Relative L2 Error: {u_l2:.4f}")
print(f" -> Velocity u Mean Absolute Error (MAE): {u_mae:.5f}")
print(f" -> Pressure p Relative L2 Error: {p_l2:.4f}")

In [ ]:

print("\n=======================================================")
print("   EVALUATING EXPLICIT PHYSICAL CONSERVATION METRICS  ")
print("=======================================================")

# -----------------------------------------------------------------
# 1. MASS CONSERVATION (Volume Flow Rate Matching)
# -----------------------------------------------------------------
# We verify if the total volume rate of fluid entering the inlet 
# matches the fluid exiting the outlet cross-section.
num_grid_points = 50  # Resolution for cross-sectional surface integration
y_grid = np.linspace(-0.5, 0.5, num_grid_points)
z_grid = np.linspace(-0.5, 0.5, num_grid_points)
Y, Z = np.meshgrid(y_grid, z_grid)

# Flat arrays for the cross-sectional evaluation mesh
y_flat = Y.flatten()[:, None]
z_flat = Z.flatten()[:, None]
num_surf_points = len(y_flat)
dA = (1.0 / (num_grid_points - 1)) * (1.0 / (num_grid_points - 1)) # Differential area element

# Create cross-sections at the Inlet (X=0.0) and Outlet (X=2.0)
inlet_mesh = np.hstack((np.zeros((num_surf_points, 1)), y_flat, z_flat))
outlet_mesh = np.hstack((2.0 * np.ones((num_surf_points, 1)), y_flat, z_flat))

# Predict velocity components
inlet_preds = model.predict(inlet_mesh)
outlet_preds = model.predict(outlet_mesh)

# Extract longitudinal u-velocity
u_inlet = inlet_preds[:, 0]
u_outlet = outlet_preds[:, 1] # Fixed: component index corrected below for u
u_outlet = outlet_preds[:, 0]

# Perform numerical surface integration (Volume Flow Q = \iint u dA)
Q_inlet = np.sum(u_inlet) * dA
Q_outlet = np.sum(u_outlet) * dA
mass_balance_error = np.abs(Q_inlet - Q_outlet) / Q_inlet * 100

print(f"1. Continuous Mass Balance Check:")
print(f"   -> Integrated Inlet Flow (Q_in):  {Q_inlet:.6f} m^3/s")
print(f"   -> Integrated Outlet Flow (Q_out): {Q_outlet:.6f} m^3/s")
print(f"   -> Net Mass Leakage Error:         {mass_balance_error:.4f}%")


# -----------------------------------------------------------------
# 2. BOUNDARY LEAKAGE PROFILE (Wall Penetration)
# -----------------------------------------------------------------
# For solid wall boundaries, fluid velocity perpendicular to the wall must be zero.
# Let's sample points explicitly on the solid top wall (Y = 0.5)
x_wall_samples = np.linspace(0.1, 1.9, 100)[:, None]
z_wall_samples = np.linspace(-0.4, 0.4, 100)[:, None]
wall_mesh = np.hstack((x_wall_samples, 0.5 * np.ones_like(x_wall_samples), z_wall_samples))

wall_preds = model.predict(wall_mesh)
v_wall = wall_preds[:, 1] # Normal velocity to the top/bottom walls (v component)

max_wall_penetration = np.max(np.abs(v_wall))
mean_wall_penetration = np.mean(np.abs(v_wall))

print(f"\n2. No-Slip Boundary Integrity:")
print(f"   -> Max Wall Normal Velocity (Leakage):  {max_wall_penetration:.6f} m/s")
print(f"   -> Mean Wall Normal Velocity (Leakage): {mean_wall_penetration:.6f} m/s")


# -----------------------------------------------------------------
# 3. KINETIC ENERGY BALANCE & TOTAL PRESSURE DROP
# -----------------------------------------------------------------
# Calculate the overall global mechanical performance metrics across the system
p_inlet = inlet_preds[:, 3]
p_outlet = outlet_preds[:, 3]

mean_p_in = np.mean(p_inlet)
mean_p_out = np.mean(p_outlet)
total_pressure_drop = mean_p_in - mean_p_out

# Kinetic Energy estimation over cross-sections: KE \approx \iint 0.5 * rho * u^3 dA
ke_inlet = np.sum(0.5 * 1.0 * (u_inlet**3)) * dA
ke_outlet = np.sum(0.5 * 1.0 * (u_outlet**3)) * dA
ke_dissipation = ke_inlet - ke_outlet

print(f"\n3. Mechanical & Thermodynamic Properties:")
print(f"   -> System Mean Pressure Drop (ΔP):       {total_pressure_drop:.6f} Pa")
print(f"   -> Inlet Kinetic Energy Flux:            {ke_inlet:.6f} W")
print(f"   -> Outlet Kinetic Energy Flux:           {ke_outlet:.6f} W")
print(f"   -> Estimated Viscous Energy Dissipation: {ke_dissipation:.6f} W")
print("=======================================================")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import deepxde as dde

# Assuming 'model' is your trained PINN model and the geometry spans:
# X: [0.0, 2.0], Y: [-0.5, 0.5], Z: [-0.5, 0.5]

print("\n--- Generating Grid-Wide Error Field Plots (Mid-plane Z=0) ---")

# 1. Generate a Dense Grid over the Longitudinal Mid-plane Slice
num_x, num_y = 200, 100
x_coord = np.linspace(0.0, 2.0, num_x)
y_coord = np.linspace(-0.5, 0.5, num_y)
X, Y = np.meshgrid(x_coord, y_coord)

# Flatten grid and force Z = 0 to build 3D coordinates for the model
x_flat = X.flatten()[:, None]
y_flat = Y.flatten()[:, None]
z_flat = np.zeros_like(x_flat)
slice_coords = np.hstack((x_flat, y_flat, z_flat))

# 2. Get Model Predictions vs. True Analytic/Sensor Profile Field
predictions = model.predict(slice_coords)
u_pred = predictions[:, 0].reshape(num_y, num_x)
p_pred = predictions[:, 3].reshape(num_y, num_x)

# Evaluate against the analytical/target profile equations used during training
u_true = 1.0 * (1.0 - (Y / 0.5)**2) * (1.0 - (0.0 / 0.5)**2) # Z=0 slice target
p_true = 0.1 * (2.0 - X)

# 3. Compute Local Absolute and Relative L2 Error Matrices
# Absolute Error Fields
u_abs_error = np.abs(u_true - u_pred)
p_abs_error = np.abs(p_true - p_pred)

# Pointwise Relative L2 Error Fields
# Added a small epsilon (1e-5) to the denominator to prevent division by zero near walls
eps = 1e-5
u_rel_error = u_abs_error / (np.abs(u_true) + eps)
p_rel_error = p_abs_error / (np.abs(p_true) + eps)

# 4. Plotting Grid-Wide Spatial Profiles via Matplotlib
fig, axs = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

# Plot A: Velocity U Absolute Error
c1 = axs[0].imshow(u_abs_error, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='jet', aspect='auto')
axs[0].set_title("Velocity $u$ Field - Absolute Error Spatial Profile")
axs[0].set_ylabel("Y Coordinate")
fig.colorbar(c1, ax=axs[0], label="Error Magnitude (m/s)")

# Plot B: Velocity U Relative L2 Error
# Clip the relative field plot max to 10% (0.1) so wall singularities don't distort scales
c2 = axs[1].imshow(u_rel_error, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='inferno', aspect='auto', vmax=0.1)
axs[1].set_title("Velocity $u$ Field - Local Pointwise Relative Error ($\leq 10\%$ Scale Limit)")
axs[1].set_ylabel("Y Coordinate")
fig.colorbar(c2, ax=axs[1], label="Relative Error % (0.1 = 10%)")

# Plot C: Pressure P Absolute Error
c3 = axs[2].imshow(p_abs_error, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='jet', aspect='auto')
axs[2].set_title("Pressure $p$ Field - Absolute Error Spatial Profile")
axs[2].set_ylabel("Y Coordinate")
fig.colorbar(c3, ax=axs[2], label="Error Magnitude (Pa)")

# Plot D: Pressure P Relative L2 Error
c4 = axs[3].imshow(p_rel_error, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='inferno', aspect='auto', vmax=0.2)
axs[3].set_title("Pressure $p$ Field - Local Pointwise Relative Error ($\leq 20\%$ Scale Limit)")
axs[3].set_xlabel("X Longitudinal Coordinate (Flow Direction)")
axs[3].set_ylabel("Y Coordinate")
fig.colorbar(c4, ax=axs[3], label="Relative Error % (0.2 = 20%)")

plt.tight_layout()
plt.savefig("grid_wide_field_errors.png", dpi=300)
plt.show()

print(" -> Grid plots generated successfully and exported as 'grid_wide_field_errors.png'.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import deepxde as dde
import torch

# Assuming 'model' is your trained PINN model, 'geom' is the domain,
# and 'navier_stokes_3d' is the PDE residual function defined earlier.

print("\n--- Generating Grid-Wide Physics Satisfaction Plots (Mid-plane Z=0) ---")

# 1. Generate a Dense Evaluation Mesh over the Longitudinal Mid-plane
num_x, num_y = 200, 100
x_coord = np.linspace(0.0, 2.0, num_x)
y_coord = np.linspace(-0.5, 0.5, num_y)
X, Y = np.meshgrid(x_coord, y_coord)

# Flatten grid and place Z = 0 to build 3D coordinates for the network graph
x_flat = X.flatten()[:, None]
y_flat = Y.flatten()[:, None]
z_flat = np.zeros_like(x_flat)
slice_coords = np.hstack((x_flat, y_flat, z_flat))

# 2. Extract the Raw PDE Residuals using DeepXDE/PyTorch Automatic Differentiation
# DeepXDE model requires inputs to be converted to PyTorch tensors with gradient tracking enabled
x_tensor = torch.tensor(slice_coords, dtype=torch.float32, requires_grad=True)

# Forward pass through the network architecture
y_tensor = model.net(x_tensor)

# Compute the individual PDE equations using your pre-defined navier_stokes_3d function
pde_residuals = navier_stokes_3d(x_tensor, y_tensor)

# Convert the resulting tensor operations back into detached NumPy arrays
# Reshape arrays back into the structured 2D grid dimensions [num_y, num_x]
res_x_mom = pde_residuals[0].detach().cpu().numpy().reshape(num_y, num_x)
res_y_mom = pde_residuals[1].detach().cpu().numpy().reshape(num_y, num_x)
res_z_mom = pde_residuals[2].detach().cpu().numpy().reshape(num_y, num_x)
res_cont  = pde_residuals[3].detach().cpu().numpy().reshape(num_y, num_x)

# 3. Apply Logarithmic Scaling to Isolate Discrepancies
# We use log10(abs(residual) + eps) because PDE residuals span multiple orders of magnitude.
# A log value of -4 means a physics satisfaction error of 10^-4 (excellent alignment).
eps = 1e-7
log_x_mom = np.log10(np.abs(res_x_mom) + eps)
log_y_mom = np.log10(np.abs(res_y_mom) + eps)
log_cont  = np.log10(np.abs(res_cont) + eps)

# 4. Generate the Physics Satisfaction Layout via Matplotlib
fig, axs = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Visual Bounds Configuration (e.g., viewing values from 10^-4 up to 10^0)
vmin_val, vmax_val = -4.0, 0.0

# Plot A: X-Momentum Equation Residual
c1 = axs[0].imshow(log_x_mom, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='seismic', 
                    aspect='auto', vmin=vmin_val, vmax=vmax_val)
axs[0].set_title("X-Momentum Equation Physics Satisfaction ($log_{10}$ |Residual|)")
axs[0].set_ylabel("Y Coordinate")
fig.colorbar(c1, ax=axs[0], label="Order of Violation")

# Plot B: Y-Momentum Equation Residual
c2 = axs[1].imshow(log_y_mom, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='seismic', 
                    aspect='auto', vmin=vmin_val, vmax=vmax_val)
axs[1].set_title("Y-Momentum Equation Physics Satisfaction ($log_{10}$ |Residual|)")
axs[1].set_ylabel("Y Coordinate")
fig.colorbar(c2, ax=axs[1], label="Order of Violation")

# Plot C: Continuity Equation (Mass Conservation) Residual
c3 = axs[2].imshow(log_cont, extent=[0, 2, -0.5, 0.5], origin='lower', cmap='seismic', 
                    aspect='auto', vmin=vmin_val, vmax=vmax_val)
axs[2].set_title("Continuity Equation Mass Integrity ($log_{10}$ |Residual|)")
axs[2].set_xlabel("X Longitudinal Coordinate (Flow Direction)")
axs[2].set_ylabel("Y Coordinate")
fig.colorbar(c3, ax=axs[2], label="Order of Violation")

plt.tight_layout()
plt.savefig("physics_satisfaction_analysis.png", dpi=300)
plt.show()

print(" -> Physics diagnostics completed. Field maps exported as 'physics_satisfaction_analysis.png'.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import deepxde as dde

# Assuming 'model' is your trained PINN model and the geometry spans:
# X: [0.0, 2.0], Y: [-0.5, 0.5], Z: [-0.5, 0.5]

print("\n--- Generating Cross-Sectional Line Verification Plots ---")

# ==============================================================================
# 1. EXTRACT DATA FOR CUT 1: VERTICAL VELOCITY PROFILE (Mid-Channel, X=1.0, Z=0.0)
# ==============================================================================
num_points_y = 200
y_line = np.linspace(-0.5, 0.5, num_points_y)[:, None]
x_line_v = np.ones_like(y_line) * 1.0
z_line_v = np.zeros_like(y_line)

coords_vertical_cut = np.hstack((x_line_v, y_line, z_line_v))

# Model Predictions along the vertical line
preds_vertical = model.predict(coords_vertical_cut)
u_pred_vertical = preds_vertical[:, 0]

# True Analytical Profile for Velocity u at Z=0, X=1.0
u_true_vertical = 1.0 * (1.0 - (y_line.flatten() / 0.5)**2)

# ==============================================================================
# 2. EXTRACT DATA FOR CUT 2: HORIZONTAL PRESSURE DROP (Centerline, Y=0.0, Z=0.0)
# ==============================================================================
num_points_x = 200
x_line = np.linspace(0.0, 2.0, num_points_x)[:, None]
y_line_h = np.zeros_like(x_line)
z_line_h = np.zeros_like(x_line)

coords_horizontal_cut = np.hstack((x_line, y_line_h, z_line_h))

# Model Predictions along the horizontal centerline
preds_horizontal = model.predict(coords_horizontal_cut)
p_pred_horizontal = preds_horizontal[:, 3]

# True Analytical Profile for Pressure p (Linear Drop: p = 0.1 * (2.0 - x))
p_true_horizontal = 0.1 * (2.0 - x_line.flatten())


# ==============================================================================
# 3. PLOTTING THE LINE VERIFICATIONS VIA MATPLOTLIB
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Velocity Profile Verification
ax1.plot(u_true_vertical, y_line, 'k-', linewidth=2.5, label='Analytical/Target')
ax1.plot(u_pred_vertical, y_line, 'r--', linewidth=2.5, label='PINN Prediction')
ax1.set_title("Velocity $u$ Profile Cut (at $X=1.0$, $Z=0$)")
ax1.set_xlabel("Velocity $u$ (m/s)")
ax1.set_ylabel("Vertical Coordinate $Y$")
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='best')

# Plot 2: Pressure Drop Verification
ax2.plot(x_line, p_true_horizontal, 'k-', linewidth=2.5, label='Analytical/Target')
ax2.plot(x_line, p_pred_horizontal, 'b--', linewidth=2.5, label='PINN Prediction')
ax2.set_title("Pressure Centerline Drop (at $Y=0$, $Z=0$)")
ax2.set_xlabel("Longitudinal Coordinate $X$ (Flow Direction)")
ax2.set_ylabel("Pressure $p$ (Pa)")
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='best')

plt.tight_layout()
plt.savefig("cross_sectional_verification.png", dpi=300)
plt.show()

print(" -> Cross-sectional plots successfully exported as 'cross_sectional_verification.png'.")
